# Projektovanje konvolucione neuralne mreže za klasifikaciju slika
## AI vs Human Generated Images

**Opis problema:** Klasifikacija slika u dve klase — slike generisane veštačkom inteligencijom (AI) i autentične slike koje su nastale fotografisanjem (Human). Model treba da nauči da prepoznaje razlike između ove dve kategorije na osnovu vizuelnih karakteristika.

## 1. Imports

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os

from keras.utils import image_dataset_from_directory
from keras.models import Sequential
from keras import layers
from keras.callbacks import EarlyStopping
from keras.losses import SparseCategoricalCrossentropy

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow version: 2.20.0
GPU available: False


## 2. Preuzimanje skupa podataka sa Kaggle

In [ ]:
import subprocess
import os

# Postavljanje Kaggle API tokena
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_fd2c5151f762387148d28c5a8dd2d08f'

DATA_DIR = './data'

if not os.path.exists(os.path.join(DATA_DIR, 'train')):
    print('Preuzimanje dataseta...')
    subprocess.run([
        'kaggle', 'datasets', 'download',
        '-d', 'alessandrasala79/ai-vs-human-generated-dataset',
        '-p', DATA_DIR, '--unzip'
    ], check=True)
    print('Dataset preuzet i raspakovan.')
else:
    print('Dataset vec postoji.')

Preuzimanje dataseta...
Dataset URL: https://www.kaggle.com/datasets/alessandrasala79/ai-vs-human-generated-dataset
License(s): apache-2.0


 56%|█████▌    | 5.42G/9.76G [04:15<03:28, 22.4MB/s]  

In [ ]:
# Pregled strukture foldera
for root, dirs, files in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    indent = ' ' * 2 * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth < 2:
        for d in dirs:
            subdir_path = os.path.join(root, d)
            n_files = len([f for f in os.listdir(subdir_path) if os.path.isfile(os.path.join(subdir_path, f))])
            print(f'{indent}  {d}/ ({n_files} fajlova)')

## 3. Učitavanje i eksploracija podataka

In [ ]:
# Parametri
IMG_SIZE = (128, 128)
BATCH_SIZE = 64
SEED = 42

# Proveriti tačnu putanju nakon preuzimanja dataseta.
# Prilagoditi TRAIN_DIR i TEST_DIR prema strukturi raspakovanih fajlova.
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
TEST_DIR = os.path.join(DATA_DIR, 'test')

# Ako dataset nema poseban test folder, koristimo validation_split
# Za slucaj da postoji samo jedan folder sa klasama:
if os.path.exists(TRAIN_DIR):
    # Train set delimo na train i validaciju
    Xtrain, Xval = image_dataset_from_directory(
        TRAIN_DIR,
        subset='both',
        validation_split=0.2,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED
    )
    
    # Test set
    if os.path.exists(TEST_DIR):
        Xtest = image_dataset_from_directory(
            TEST_DIR,
            image_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            shuffle=False
        )
    else:
        print('Test folder ne postoji - koristimo validacioni skup kao test.')
        Xtest = Xval
else:
    # Ako je sve u jednom folderu
    main_dir = DATA_DIR
    Xtrain, Xval = image_dataset_from_directory(
        main_dir,
        subset='both',
        validation_split=0.3,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED
    )
    Xtest = Xval

classes = Xtrain.class_names
num_classes = len(classes)
print(f'Klase: {classes}')
print(f'Broj klasa: {num_classes}')

In [ ]:
# Prikaz jednog primerka iz svake klase
fig, axes = plt.subplots(1, num_classes, figsize=(5 * num_classes, 5))
if num_classes == 1:
    axes = [axes]

shown_classes = set()
for images, labels in Xtrain:
    for i in range(len(labels)):
        label = labels[i].numpy()
        if label not in shown_classes:
            ax = axes[label]
            ax.imshow(images[i].numpy().astype('uint8'))
            ax.set_title(classes[label], fontsize=14)
            ax.axis('off')
            shown_classes.add(label)
        if len(shown_classes) == num_classes:
            break
    if len(shown_classes) == num_classes:
        break

plt.suptitle('Jedan primerak iz svake klase', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram raspodele odbiraka po klasama
class_counts = {c: 0 for c in classes}

for _, labels in Xtrain.unbatch():
    class_counts[classes[labels.numpy()]] += 1

print('Broj odbiraka po klasama (trening skup):')
for c, count in class_counts.items():
    print(f'  {c}: {count}')

plt.figure(figsize=(8, 5))
plt.bar(class_counts.keys(), class_counts.values(), color=['#4C72B0', '#DD8452'][:num_classes])
plt.xlabel('Klasa')
plt.ylabel('Broj odbiraka')
plt.title('Raspodela odbiraka po klasama (trening skup)')
for i, (c, v) in enumerate(class_counts.items()):
    plt.text(i, v + 10, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Provera balansiranosti
counts = list(class_counts.values())
ratio = max(counts) / min(counts) if min(counts) > 0 else float('inf')
print(f'\nOdnos najvece/najmanje klase: {ratio:.2f}')
if ratio < 1.5:
    print('Podaci su priblizno balansirani.')
else:
    print('Podaci NISU balansirani - potrebno je primeniti tehniku balansiranja.')

In [ ]:
# Racunanje class_weight u slucaju nebalansiranih klasa
from sklearn.utils.class_weight import compute_class_weight

all_labels = []
for _, labels in Xtrain.unbatch():
    all_labels.append(labels.numpy())
all_labels = np.array(all_labels)

weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weight = dict(enumerate(weights))
print(f'Class weights: {class_weight}')

## 4. Predprocesiranje i augmentacija podataka

Primenjene transformacije:
- **Skaliranje (Rescaling):** Vrednosti piksela se normalizuju sa opsega [0, 255] na [0, 1] kako bi se ubrzala konvergencija treninga.
- **Horizontalno okretanje (RandomFlip):** Slika se nasumično okreće po horizontalnoj osi, čime se povećava raznovrsnost trening skupa.
- **Nasumična rotacija (RandomRotation):** Slika se rotira za nasumični ugao, što pomaže modelu da bude otporniji na orijentaciju objekata.
- **Nasumični zum (RandomZoom):** Slika se nasumično zumira, simulirajući različite udaljenosti kamere.

Ove augmentacije se primenjuju samo tokom treninga i pomažu u smanjenju preobucenosti (overfitting).

In [ ]:
# Augmentacija podataka
data_augmentation = Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
])

# Prikaz augmentiranih slika
plt.figure(figsize=(12, 6))
for images, _ in Xtrain.take(1):
    img = images[0]
    for i in range(8):
        aug_img = data_augmentation(tf.expand_dims(img, 0))
        plt.subplot(2, 4, i + 1)
        plt.imshow(aug_img[0].numpy().astype('uint8'))
        plt.axis('off')
plt.suptitle('Primeri augmentacije jedne slike', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Definisanje CNN modela

Projektovana je konvoluciona neuralma mreža sa 4 konvoluciona bloka.

**Obrazloženje izbora:**
- **Kriterijumska funkcija:** `SparseCategoricalCrossentropy` — standardni izbor za klasifikaciju u više klasa gde su labele celobrojne.
- **Funkcija aktivacije neurona:** `ReLU` u skrivenim slojevima (rešava problem nestajućeg gradijenta, računski efikasna), `Softmax` u izlaznom sloju (daje distribuciju verovatnoća po klasama).
- **Optimizacija:** `Adam` — adaptivna metoda optimizacije koja kombinuje prednosti AdaGrad i RMSProp algoritama, dobro funkcioniše sa podrazumevanim hiperparametrima.

**Zaštita od preobučavanja (overfitting):**

Preobučavanje je pojava kada model previše dobro nauči trening podatke, uključujući šum i specifičnosti, pa ne uspeva da generalizuje na nove, neviđene podatke. To se manifestuje kroz veliku razliku između performansi na trening i validacionom/test skupu.

Primenjene metode zaštite:
- **Dropout (0.3):** Tokom treninga, nasumično isključuje 30% neurona, čime se sprečava ko-adaptacija i forsira redundantna reprezentacija.
- **Batch Normalization:** Normalizuje aktivacije unutar svakog mini-batch-a, stabilizuje trening i ima blagi regularizacioni efekat.
- **Early Stopping:** Prekida trening kada se validaciona metrika prestane poboljšavati, čime se sprečava nepotrebno treniranje koje vodi ka preobučavanju.

In [ ]:
def cnn_model(num_classes):
    model = Sequential([
        # Augmentacija i skaliranje (nova instanca, nezavisna od globalne)
        Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.15),
            layers.RandomZoom(0.1),
        ]),
        layers.Rescaling(1./255),

        # Konvolucioni blok 1
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Konvolucioni blok 2
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Konvolucioni blok 3
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Konvolucioni blok 4
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Klasifikaciona glava
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss=SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model

In [ ]:
model = cnn_model(num_classes)
model.summary()

## 6. Treniranje modela

In [ ]:
es = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    Xtrain,
    epochs=30,
    validation_data=Xval,
    callbacks=[es],
    class_weight=class_weight,
    verbose=1
)

## 7. Grafik promena performansi tokom epoha treniranja

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Tacnost
ax1.plot(history.history['accuracy'], label='Trening')
ax1.plot(history.history['val_accuracy'], label='Validacija')
ax1.set_title('Tačnost po epohama')
ax1.set_xlabel('Epoha')
ax1.set_ylabel('Tačnost')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Gubitak
ax2.plot(history.history['loss'], label='Trening')
ax2.plot(history.history['val_loss'], label='Validacija')
ax2.set_title('Gubitak po epohama')
ax2.set_xlabel('Epoha')
ax2.set_ylabel('Gubitak')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Evaluacija modela

In [ ]:
# Predikcije na test skupu
y_true_test = np.array([])
y_pred_test = np.array([])
for img, lab in Xtest:
    y_true_test = np.concatenate([y_true_test, lab.numpy()])
    y_pred_test = np.concatenate([y_pred_test, np.argmax(model.predict(img, verbose=0), axis=1)])

print(f'Tačnost modela na test skupu: {100 * accuracy_score(y_true_test, y_pred_test):.2f}%')
print()
print('Detaljan izveštaj:')
print(classification_report(y_true_test, y_pred_test, target_names=classes))

In [ ]:
# Predikcije na trening skupu
y_true_train = np.array([])
y_pred_train = np.array([])
for img, lab in Xtrain:
    y_true_train = np.concatenate([y_true_train, lab.numpy()])
    y_pred_train = np.concatenate([y_pred_train, np.argmax(model.predict(img, verbose=0), axis=1)])

print(f'Tačnost modela na trening skupu: {100 * accuracy_score(y_true_train, y_pred_train):.2f}%')

## 9. Matrica konfuzije

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Matrica konfuzije - trening skup
cm_train = confusion_matrix(y_true_train, y_pred_train, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm_train, display_labels=classes).plot(ax=ax1)
ax1.set_title('Matrica konfuzije - Trening skup')

# Matrica konfuzije - test skup
cm_test = confusion_matrix(y_true_test, y_pred_test, normalize='true')
ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=classes).plot(ax=ax2)
ax2.set_title('Matrica konfuzije - Test skup')

plt.tight_layout()
plt.show()

## 10. Primeri dobro i loše klasifikovanih slika

In [ ]:
# Prikupljanje slika, pravih labela i predikcija sa test skupa
all_images = []
all_true = []
all_pred = []

for img_batch, lab_batch in Xtest:
    preds = np.argmax(model.predict(img_batch, verbose=0), axis=1)
    for i in range(len(lab_batch)):
        all_images.append(img_batch[i].numpy().astype('uint8'))
        all_true.append(lab_batch[i].numpy())
        all_pred.append(preds[i])

all_true = np.array(all_true)
all_pred = np.array(all_pred)

correct_idx = np.where(all_true == all_pred)[0]
incorrect_idx = np.where(all_true != all_pred)[0]

print(f'Ukupno tačno klasifikovanih: {len(correct_idx)}')
print(f'Ukupno pogrešno klasifikovanih: {len(incorrect_idx)}')

In [ ]:
# Dobro klasifikovane slike
n_show = min(8, len(correct_idx))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
sample = np.random.choice(correct_idx, n_show, replace=False)
for i, idx in enumerate(sample):
    ax = axes[i // 4][i % 4]
    ax.imshow(all_images[idx])
    ax.set_title(f'Stvarno: {classes[all_true[idx]]}\nPredikcija: {classes[all_pred[idx]]}', fontsize=10)
    ax.axis('off')
plt.suptitle('Primeri DOBRO klasifikovanih slika', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Lose klasifikovane slike
n_show = min(8, len(incorrect_idx))
if n_show > 0:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    sample = np.random.choice(incorrect_idx, n_show, replace=False)
    for i, idx in enumerate(sample):
        ax = axes[i // 4][i % 4]
        ax.imshow(all_images[idx])
        ax.set_title(f'Stvarno: {classes[all_true[idx]]}\nPredikcija: {classes[all_pred[idx]]}', fontsize=10, color='red')
        ax.axis('off')
    # Sakrij prazne subplotove
    for i in range(n_show, 8):
        axes[i // 4][i % 4].axis('off')
    plt.suptitle('Primeri POGREŠNO klasifikovanih slika', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Nema pogrešno klasifikovanih slika!')